This notebook is to run word frequency analyses on my data, which I obtained by pinging the Congress.gov API, detailed in my data exploration notebook.

In [1]:
import spacy
from collections import Counter
import pandas as pd

# Load spaCy's English model
nlp = spacy.load("en_core_web_sm")


In [2]:
df_house119 = pd.read_csv('house119_enviromeetings.csv')

In [3]:
df_house118 = pd.read_csv('house118_enviromeetings.csv')

In [4]:
df_house117 = pd.read_csv('house117_enviromeetings.csv')

In [5]:
df_house116 = pd.read_csv('house116_enviromeetings.csv')

In [6]:
df_house115 = pd.read_csv('house115_enviromeetings.csv')

In [7]:
df_house114 = pd.read_csv('house114_enviromeetings.csv')

In [8]:
df_senate119 = pd.read_csv('senate119_enviromeetings.csv')

In [9]:
df_senate118 = pd.read_csv('senate118_enviromeetings.csv')

In [10]:
df_senate117 = pd.read_csv('senate117_enviromeetings.csv')

In [11]:
df_senate116 = pd.read_csv('senate116_enviromeetings.csv')

In [12]:
#df_house110, df_house111, df_house112, df_house113, 

senate_dataframes = [df_senate116, df_senate117, df_senate118, df_senate119]
house_dataframes = [df_house114, df_house115, df_house116, df_house117, df_house118, df_house119]

Make lists of all of the titles to run my frequency analysis on

1 for House meetings in R controlled congresses
1 for House meetings in D controlled congresses
1 for Senate meetings in R controlled congresses
1 for Senate meetings in D controlled congresses

In [13]:
# empty lists to fill for all the titles of meetings unde R controlled Congresses and under D controlled Congresses  

meetingnames_R_house_controlled = []
meetingnames_D_house_controlled = []

meetingnames_R_senate_controlled = []
meetingnames_D_senate_controlled = []


Now fill them, by running through each Congress and appending to the appropriate list


In [14]:
## for House 

house_D_controlled = [117, 116, 111, 110] # lists of the Congresses congrolled by Democrats and Republicans
house_R_controlled = [119, 118, 115, 114, 113, 112]

for df in house_dataframes:

    congress = df['congress'].iloc[0]

    if congress in house_D_controlled: # if the Congress number is D controlled 
        for title in df['title']: # then run through each item in the list
            print(title)
            meetingnames_D_house_controlled.append(title)

    elif congress in house_R_controlled: # if the Congress number is r controlled 
        for title in df['title']:
            meetingnames_R_house_controlled.append(title)

#print it out
print('House:')
print(len(meetingnames_D_house_controlled))
print(len(meetingnames_R_house_controlled))

Ocean Climate Action: Solutions to the Climate Crisis. Legislative hearing on H.R. 8632, the Ocean-Based Climate Solutions Act, and H.R. 3548, H.R. 3919, H.R. 4093, H.R. 5390, H.R.5589, H.R.7387, H.R. 8253, H.R.8627
Environmental Justice for Coastal Communities: Examining Inequities in Federal Grantmaking
“Generating Equity: Improving Clean Energy Access and Affordability” - Virtual Hearing
Markup of HR 244, HR 733, HR 970, HR 1248, HR 1964, HR 3225, HR 3651, HR 4135, HR 4139, HR 4840, HR 5153, HR 5458, HR 5459, HR 5472, HR 5598, HR 5852, HR 7098, HR 7099, S. 212
H.R. 5986,  To restore, reaffirm, and reconcile environmental justice and civil rights, provide for the establishment of the Interagency Working Group on Environmental Justice Compliance and Enforcement, and for other purposes. “Environmental Justice for All Act”.
Legislative Hearing on H.R. 7565 and H.R. 8255
"Puerto Rico’s Options for a Non-Territory Status and What Should Happen After November’s Plebiscite"
"Health Care Lif

In [15]:
# for Senate

senate_D_controlled = [118, 117, 113, 112, 111, 110] # lists of the Congresses congrolled by Democrats and Republicans
senate_R_controlled = [119, 116, 115, 114]

for df in senate_dataframes:

    congress = df['congress'].iloc[0]

    if congress in senate_D_controlled: # if the Congress number is D controlled 
        for title in df['title']: # then run through each item in the list
            meetingnames_D_senate_controlled.append(title)

    elif congress in senate_R_controlled: # if the Congress number is r controlled 
        for title in df['title']:
            meetingnames_R_senate_controlled.append(title)

# print it out

print('Senate:')
print(len(meetingnames_D_senate_controlled))
print(len(meetingnames_R_senate_controlled))

Senate:
183
163


Combine all items in list in a format spacy likes

In [16]:
meetingnames_R_house_controlled_forspacy = " ".join(meetingnames_R_house_controlled)

In [17]:
meetingnames_D_house_controlled_forspacy = " ".join(meetingnames_D_house_controlled)

In [18]:
meetingnames_R_senate_controlled_forspacy = " ".join(meetingnames_R_senate_controlled)

In [19]:
meetingnames_D_senate_controlled_forspacy = " ".join(meetingnames_D_senate_controlled)

Analyze the Rs and Ds

Define phrases to exclude

In [ ]:
custom_exclude = {
    # process words
    'an oversight hearing', 'a discussion draft', 'oversight field hearing',  # i am keeping just 'oversight' 
    'leg hrg', 'tuesday', 'wednesday', 'thursday', 'oversight hearing', 'hearings', 'a hearing',
    'longworth house office building', 'statements', 'business meeting', 'september', 'opening', 'november', 'certain documents', '_', 'the 1324 longworth house office building',
    '\'" legislative hearing', 'energy and commerce committee vote', 'the establishment', 'full committee markup', 'the development',  'the nominations', 'the nomination', 'inquiry',
    'the subcommittee', 'subcommittee markup', 'this notice', 'other purposes', 'discussion', 'legislative hearing', 'the nature',
    '10:00 a.m.', '12:00 p.m.', '5:00 p.m.', '4:00 p.m.',
    'multiple bills',
    'h.r.', '•\th.r.', '• h.r.', '”\r\n•\th.r.', 'hearings', '”\r\n• h.r.', '•h.r.', '” h.r.', 'hr', 
    
    # bill words
    'that act', 'section', 'an original bill', 'the amendment', 'amendments', 'legislation', 'a markup beginning', 'the conveyance', 'fiscal year', 'the implementation', 'legislative proposals',
    's.', 'the status', 'title', 'a bill', 'discussion draft', "the president's proposed budget request", 'markup', 'a substitute', 'the following bills', 'the amendments', 'amendment', '\" legislative hearing'

    # themed words but for non target topics (note that some of these committees are also health related)
    'agriculture', 'health care costs', 'covid-19', 'the covid-19 pandemic', 'patients', 'the opioid crisis', 'biomedicine'

    'way' # because I want to include "rights of way" and "right of way" as a complete phrase

    # broad actors
    'the secretary', 'deputy secretary'
    'secretary',
    'the committee',
    'the house',
    'the senate',
    'house',
    'senate'

}

In [21]:
custom_include = {

    # CLIMATE CHANGE

    'climate change',
    'climate crisis',
    'climate emergency',
    'climate',
    'reliability',

    #pollution

    'greenhouse gas', 'ghg',
    'methane', 'carbon dioxide', 'co2',
    'particulate matter', 'pfas', 'forever chemicals', 'superfund'

    # ENERGY

    # fossil 
    'fossil,'
    'crude oil',
    'natural gas',
    'coal',
    'petroleum',
    'liquefied natural gas', 
    'fracking',
    'pipelines',
    'refineries',
    'drilling permits',
    'federal lands', 
    'strategic petroleum reserve',
    'hydrocarbons',
    'offshore drilling',
    'leasing',
    'royalties',
    'lng', 'liquefied natural gas',
    
    # renewable 
    'renewable energy',
    'clean energy',
    'wind',
    'solar',
    'nuclear energy'

    # controverical renewbale'
    'hydroelectric',
    'geothermal',
    'hydrogen','green hydrogen', 'clean hydrogen', 
    'biofuels',
    'ethanol', 
    'biomass',
    'carbon capture and sequestration', 'ccs', 'carbon capture and storage', 'carbon capture', 'direct air capture', 'dac', 
    'small modular reactors', 'smrs', 'advanced nuclear',
    'fusion', 'sustainable aviation fuel', 'saf',
    
    # grid & supply chain infrastructure
    'transmission lines', 'interconnection queue',
    'ferc',
    
    # other electricity / grid
    
    'critical minerals', 'supply chain'
    'electric vehicles', 'evs', 'batteries', 'charging stations', 'ev charging',
    'electric', 'electricity',
    'heat pumps', 'rights of way', 'right of way'

    # WEATHER / EXTREME WEATEHR EVENTS

    'extreme weather',
    'wildfires', 'wildfire'
    'drought', 'droughts'
    'sea level rise',
    'flooding', 'floods', 'national flood insurance program', 'nfip'

    # ENERGY MESSAGING

    # progressive / democratic framing
    'climate justice', 'environmental justice', 'existential threat', 'polluters', 'big oil', 'sustainability', 'just transition',
    'clean energy economy', 'decarbonization', 'energy transition', 'targets', 'net-zero', 'net zero', 'zero-emission',
    
    # conservative / republican framing
    'regulatory overreach', 'bureaucrating stringency', 'war on coal', 'green new deal', 
    'unreliable', 'reliable', 'energy tax', 'mandates', 'job', 'jobs', 'china', 'chinese'
    'global competitiveness', 'american energy', 'free market', 'innovation over regulation'
    'all of the above', 'dominance', 'abundance', 'costs', 'prices', 'emergency', 'independence', 'security', 
    
    # other rules and policies

    'environmental protection agency', 'epa',
    'department of energy', 'doe'
    'regulation', 'deregulation'
    'tax credits', 'subsidies',
    'permitting reform',
    'weatherization',
    'carbon tax', 'carbon pricing', 'cap and trade', 'esg',
    'sec disclosure', 'green bonds', 'production tax credit', 'ptc', 'investment tax credit', 'itc',
    'clean air act', 'clean water act'
    'fish and wildlife service', 'fish & wildlife service',
    'inflation reduction act', 'ira',
    'bipartisan infrastructure law', 'bil'
    'energy bill of 2020', '2020 energy law',
    'historic preservation act', #some of these, as this one, were developed because spacy was cutting off the end of some of these phrases
    'bureau of land management', 'blm' # same with this one, didn't want just 'bureau'

    # international treaties

    'paris agreement', 'paris accord', 'kyoto protocol', 'cop', 'united nations', 'un'

    # GEOGRAPHY 

    # all state names and abbreviations

    'alabama', 'al',
    'alaska', 'ak',
    'arizona', 'az',
    'arkansas', 'ar',
    'california', 'ca',
    'colorado', 'co',
    'connecticut', 'ct',
    'delaware', 'da',
    'district of Columbia', 'dc', 'd.c.',
    'florida', 'fl',
    'georgia', 'ga',
    'hawaii', 'hi',
    'idaho', 'id',
    'illinois', 'il',
    'indiana', 'in',
    'iowa', 'ia',
    'kansas', 'ks',
    'kentucky', 'ky',
    'louisiana', 'la',
    'maine', 'me',
    'maryland', 'md',
    'massachusetts', 'ma',
    'michigan', 'mi',
    'minnesota', 'mn',
    'mississippi', 'ms',
    'missouri', 'mo',
    'montana', 'mt',
    'nebraska', 'ne',
    'nevada', 'nv',
    'new hampshire', 'nh',
    'new jersey', 'nj',
    'new mexico', 'nm',
    'new york', 'ny',
    'north carolina', 'nc',
    'north dakota', 'nd',
    'ohio', 'oh',
    'oklahoma', 'ok',
    'oregon', 'or',
    'pennsylvania', 'pa',
    'rhode island', 'ri',
    'south carolina', 'sc',
    'south dakota', 'sd',
    'tennessee', 'tn',
    'texas', 'tx',
    'utah', 'ut',
    'vermont', 'vt',
    'virginia', 'va',
    'washington', 'wa',
    'west virginia', 'wv',
    'wisconsin', 'wi',
    'wyoming', 'wy',

    # other geographic words 

    'southern', 'south',
    'northern', 'north', 'northeast','northeastern', 
    'midwest',
    'pacific northwest',
    'west', 'western', 
    'coastal',
    'gulf coast', 'permian basin', 'gulf of mexico', 'gulf of america'
    'rust belt',
    'sun belt',
    'bible belt',
    'new england',
    'great lakes',
    'appalachia',
    'rural',
    'urban', 'cities', 'city',
    'suburban', 'suburb',

}

R meetings House

In [22]:
doc = nlp(meetingnames_R_house_controlled_forspacy)

noun_phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks] # Extract grammatical noun phrases, lowercased and stripped of whitespace
# Add the custom included items
noun_phrases.extend(custom_include)

# Filter out specific phrases manually
exclude = custom_exclude

# Filter in specific phrases

text = meetingnames_R_house_controlled_forspacy.lower()
included = [
    phrase for phrase in custom_include
    if phrase in text
    
]

# add in those phrases with spacy's phrases:
the_rest = [p for p in noun_phrases if p not in exclude]

phrase_counts = Counter(the_rest)

df_top_R_house_phrases = pd.DataFrame(phrase_counts.most_common(150), columns=['Noun Phrase', 'Frequency'])

D meetings House

In [23]:
doc = nlp(meetingnames_D_house_controlled_forspacy)

noun_phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks] # Extract grammatical noun phrases, lowercased and stripped of whitespace
# Add the custom included items
noun_phrases.extend(custom_include)
# Filter out specific phrases manually
exclude = custom_exclude

# Filter in specific phrases

text = meetingnames_D_house_controlled_forspacy.lower()
included = [
    phrase for phrase in custom_include
    if phrase in text
]

# add in those phrases with spacy's phrases:
the_rest = [p for p in noun_phrases if p not in exclude]

phrase_counts = Counter(the_rest)

df_top_D_house_phrases = pd.DataFrame(phrase_counts.most_common(150), columns=['Noun Phrase', 'Frequency'])


R meetings Senate

In [24]:
doc = nlp(meetingnames_R_senate_controlled_forspacy)

noun_phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks] # Extract grammatical noun phrases, lowercased and stripped of whitespace
# Add the custom included items
noun_phrases.extend(custom_include)
# Filter out specific phrases manually
exclude = custom_exclude

# Filter in specific phrases

text = meetingnames_R_senate_controlled_forspacy.lower()
included = [
    phrase for phrase in custom_include
    if phrase in text
]

# add in those phrases with spacy's phrases:
the_rest = [p for p in noun_phrases if p not in exclude]

phrase_counts = Counter(the_rest)

df_top_R_senate_phrases = pd.DataFrame(phrase_counts.most_common(150), columns=['Noun Phrase', 'Frequency'])


D meetings Senate

In [25]:
doc = nlp(meetingnames_D_senate_controlled_forspacy)

noun_phrases = [chunk.text.lower().strip() for chunk in doc.noun_chunks] # Extract grammatical noun phrases, lowercased and stripped of whitespace
# Add the custom included items
noun_phrases.extend(custom_include)

# Filter out specific phrases manually
exclude = custom_exclude

# Filter in specific phrases

text = meetingnames_D_senate_controlled_forspacy.lower()
included = [
    phrase for phrase in custom_include
    if phrase in text
]

# add in those phrases with spacy's phrases:
the_rest = [p for p in noun_phrases if p not in exclude]

phrase_counts = Counter(the_rest)

df_top_D_senate_phrases = pd.DataFrame(phrase_counts.most_common(150), columns=['Noun Phrase', 'Frequency'])

Save dataframes

In [26]:
df_top_R_house_phrases.to_csv('top_R_house_phrases.csv')

In [27]:
df_top_D_house_phrases.to_csv('top_D_house_phrases.csv')

In [28]:
df_top_R_senate_phrases.to_csv('top_R_senate_phrases.csv')

In [29]:
df_top_D_senate_phrases.to_csv('top_D_senate_phrases.csv')

In [30]:
df_top_D_senate_phrases.head()

,Noun Phrase,Frequency
0,the state,70
1,the secretary,64
2,the interior,55
3,energy,47
4,the department,27


In [31]:
df_top_D_house_phrases.head()

,Noun Phrase,Frequency
0,the interior,13
1,the secretary,13
2,the department,11
3,representatives,11
4,america,9
